In [9]:
import math
class Scalar:
    def __init__(self, data, _children = ()):
        self.grad = 0.0
        self.data = data
        self._prev = set(_children)
        self._backward = lambda:None

    def __repr__(self):
        return f"data = {self.data}"
    
    #operations

    def __add__(self, other):
        if isinstance(other, Scalar):
            other = other
        else:
            other = Scalar(other)

        out = Scalar(self.data + other.data, (self, other)) #same scalar object with self and other as child

        def _backward():
            self.grad += 1 * out.grad #df/dx = dz/dx * df/dz
            other.grad += 1 * out.grad
        out._backward = _backward

        return out
    
    def __pow__(self,other):
        if isinstance(other,Scalar):
            other = other
        else:
            other = Scalar(other)
        out = Scalar(self.data**other.data,(self,))

        def _backward():
            self.grad += other.data * self.data ** (other.data-1) * out.grad
        out._backward = _backward

        return out
    
    def __truediv__(self,other):
        if isinstance(other,Scalar):
            other = other
        else:
            other = Scalar(other)

        out = self * other**-1 #reuse multiplication
        return out

    
    def __mul__(self,other):
        if isinstance(other, Scalar):
            other = other
        else:
            other = Scalar(other)

        out = Scalar(self.data * other.data, (self, other))

        def _backward():
            self.grad +=  other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __neg__(self):
        out = self * Scalar(-1) #just using already exisitng multiplication
        return out 
    
    def __sub__(self, other):
        out = self + (-other)
        return out
    
    def exp(self):
        out = Scalar(math.exp(self.data),(self, ))

        def _backward():
            self.grad += math.exp(self.data) * out.grad
        out._backward = _backward

        return out
    
    def log(self):
        out = Scalar(math.log(self.data), (self, ))

        def _backward():
            self.grad += 1 / self.data * out.grad
        out._backward = _backward

        return out
    
    #activation functions
    def tanh(self):
        
        out = Scalar(math.tanh(self.data), (self,));
    
        def _backward():
            a = math.tanh(self.data)
            self.grad += (1 - a*a) * out.grad #sec^2 theta equals 1-tan^2 theta
        out._backward = _backward 
        return out
    
    def relu(self):
        x = self.data
        x = x if x>0 else 0

        out = Scalar(x, (self,));

        def _backward():
            if self.data > 0:
                self.grad+=out.grad
            else:
                self.grad += 0
        out._backward = _backward
        return out
    
    
    #backward prop logic
    def backward(self):
        self.grad = 1.0
        visited = set()
        order =[]

        def traverse(i):
            if i not in visited:
                visited.add(i)
                for j in i._prev:
                    traverse(j)
                order.append(i)
        traverse(self)

        for i in reversed(order):
            i._backward()
                
        




In [10]:
x = Scalar(2)
y = Scalar(3)
z = x.log()*y + Scalar(10)
a = z-z*2
a.backward()
print(x.grad, y.grad,z.grad)


-1.5 -0.6931471805599453 -1.0
